In [ ]:
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr tesseract-ocr-fas
!pip install -q pymupdf pytesseract pillow langchain langchain-community

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:

from google.colab import drive
import fitz
import pytesseract
from PIL import Image
import io
import re
from langchain_core.documents import Document

drive.mount('/content/drive')


#  OCRهر صفحه
def ocr_page(page, dpi=300, top_crop=0.08, bottom_crop=0.10):
    pix = page.get_pixmap(dpi=dpi)
    img = Image.open(io.BytesIO(pix.tobytes("png")))
    w, h = img.size
    cropped = img.crop((0, int(h * top_crop), w, int(h * (1 - bottom_crop))))
    return pytesseract.image_to_string(cropped, lang="fas").strip()


NOISE_PATTERNS = [
    r"کتابخانة?\s*سایت\s*نسیم\s*مطهر",
    r"nasimemotahar\.com",
    r"لطفاً?\s*نواقص.*?اطلاع\s*دهید\.?",
]

def remove_footnotes(text):
    """
    پاورقی‌ها همیشه در انتهای متن اصلی صفحه می‌آیند و با کلمه «پاورقی»
    شروع می‌شوند. چون OCR گاهی این کلمه را «پاورفی» یا با فاصله عجیب
    می‌خواند، الگوی منعطف به کار می‌بریم. هر چه بعد از این کلمه بیاید
    (شماره‌ها و توضیحات پاورقی) حذف می‌شود.
    """
    match = re.search(r"پاور\s?[قف]\s?ی", text)
    if match:
        text = text[: match.start()]
    return text.strip()

def basic_clean(text):
    for pattern in NOISE_PATTERNS:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE)
    text = re.sub(r"[\(\uFF08]\s?[۰-۹0-9]\s?[\)\uFF09]", "", text)
    lines = [ln for ln in text.split("\n") if len(ln.strip()) > 3]
    text = "\n".join(lines)
    return re.sub(r"\n{3,}", "\n\n", text).strip()


def is_heading(line):
    """
    هیوریستیک ساده: تیترها معمولاً کوتاه‌اند (۱ تا ۶ کلمه) و برخلاف
    جملات عادی، با نقطه یا علائم پایان جمله تمام نمی‌شوند.
    این روش کامل نیست (ممکن است جمله کوتاه را اشتباه تیتر تشخیص دهد)
    ولی برای جدا نگه داشتن تیترها از متن اصلی کافی و قابل اتکاست.
    """
    s = line.strip()
    if not s:
        return False
    words = s.split()
    return (1 <= len(words) <= 6) and not s.endswith((".", "؟", "!", "،", ":"))

def split_into_segments(text):
    """
    متن یک صفحه را به لیستی از (عنوان_بخش, متن_بخش) تقسیم می‌کند.
    عنوان بخش دیگر داخل متن اصلی نمی‌ماند، بلکه به‌عنوان متادیتای
    جدا (section) نگه‌داری می‌شود تا در پاسخ نهایی گم و قاطی نشود.
    """
    lines = text.split("\n")
    segments = []
    current_title = None
    current_body = []
    for ln in lines:
        s = ln.strip()
        if is_heading(s):
            if current_body:
                segments.append((current_title, " ".join(current_body)))
            current_title = s
            current_body = []
        else:
            current_body.append(s)
    if current_body:
        segments.append((current_title, " ".join(current_body)))
    return segments


#   چانک‌بندی بر اساس مرز جمله
def split_sentences(text):
    """جدا کردن جملات فارسی بر اساس علائم پایان جمله."""
    sentences = re.split(r"(?<=[.؟!])\s+", text)
    return [s.strip() for s in sentences if s.strip()]

def chunk_by_sentences(text, target_size=500, overlap_sentences=1):
    """
    جملات را کنار هم می‌چیند تا اندازه چانک به target_size برسد.
    چون همیشه با مرز جمله شروع و تمام می‌شود، دیگر متن از وسط
    جمله بریده نمی‌شود. overlap_sentences باعث می‌شود آخرین جمله
    یک چانک، اولین جمله چانک بعدی هم باشد تا زمینه (context) حفظ شود.
    """
    sentences = split_sentences(text)
    if not sentences:
        return []
    chunks = []
    current, current_len = [], 0
    for sent in sentences:
        current.append(sent)
        current_len += len(sent)
        if current_len >= target_size:
            chunks.append(" ".join(current))
            current = current[-overlap_sentences:] if overlap_sentences else []
            current_len = sum(len(s) for s in current)
    if current and (not chunks or " ".join(current) != chunks[-1]):
        chunks.append(" ".join(current))
    return chunks


#   نگاشت فصل‌ها
CHAPTER_MAP = {
    7: "انسان و حیوان",
    19: "علم و ایمان",
    33: "ایمان مذهبی",
    47: "مکتب، ایدئولوژی",
    63: "اسلام مکتب جامع و همه جانبه",
    71: "منابع تفکر در اسلام",
}

def get_chapter(page_number, chapter_map=CHAPTER_MAP):
    chapter = None
    for start in sorted(chapter_map.keys()):
        if page_number >= start:
            chapter = chapter_map[start]
        else:
            break
    return chapter


def merge_across_pages(page_segments):
    """
    page_segments: لیستی از (page_number, section_title, body_text) به
    ترتیب صفحات کتاب.

    مشکل: وقتی یک پاراگراف بین دو صفحه نصف می‌شود، صفحه بعد بدون تیتر
    شروع می‌شود و جمله‌اش ادامه‌ی جمله ناتمام صفحه قبل است. اینجا
    تشخیص می‌دهیم: اگر متن بخش قبلی با نقطه/علامت پایان جمله تمام
    نشده و بخش بعدی هم تیتر ندارد (یعنی صرفاً ادامه است، نه فصل/بخش
    جدید)، دو بخش را به هم می‌چسبانیم. متادیتای صفحه از روی صفحه‌ی
    شروعِ بخش (نه پایانش) نگه‌داشته می‌شود.
    """
    merged = []
    for pn, title, body in page_segments:
        prev_incomplete = (
            merged
            and title is None
            and merged[-1][2].strip()
            and merged[-1][2].strip()[-1] not in ".؟!"
        )
        if prev_incomplete:
            prev_pn, prev_title, prev_body = merged[-1]
            merged[-1] = (prev_pn, prev_title, prev_body + " " + body)
        else:
            merged.append((pn, title, body))
    return merged


def process_book(pdf_path, book_name, start_page=5, end_page=None, dpi=300,
                  target_chunk_size=500, overlap_sentences=1):
    """
    start_page=5 (ایندکس صفر-پایه) دقیقاً همان صفحه‌ای است که فصل
    «انسان و حیوان» در این کتاب شروع می‌شود (شماره چاپی ۷).

    نکته مهم درباره شماره صفحه: بین ایندکس PDF و شماره چاپی روی
    کتاب یک صفحه اختلاف است (به‌خاطر صفحات مقدماتی بدون شماره:
    جلد، فهرست، صفحه خالی). با بررسی دستی مشخص شد ایندکس ۵ = صفحه
    چاپی ۷، پس فرمول صحیح: page_number = i + 2 (نه i + 1).
    اگر بعداً کتاب دیگری با همین pipeline پردازش کردی، حتماً این
    افست را دوباره برای آن فایل چک کن (ممکن است متفاوت باشد).
    """
    doc = fitz.open(pdf_path)
    end_page = end_page or len(doc)

    print(f"در حال OCR کتاب: {book_name} ({end_page - start_page} صفحه)...")
    page_segments = []  # (page_number, section_title, body_text) برای همه صفحات
    for i in range(start_page, end_page):
        page_number = i + 2  # مطابق شماره چاپی کتاب
        raw_text = ocr_page(doc[i], dpi=dpi)
        text = remove_footnotes(raw_text)
        text = basic_clean(text)
        if not text:
            continue
        for section_title, body in split_into_segments(text):
            page_segments.append((page_number, section_title, body))
        if (i - start_page + 1) % 10 == 0:
            print(f"  ...{i - start_page + 1} صفحه پردازش شد")
    doc.close()

    merged_segments = merge_across_pages(page_segments)

    all_chunks = []
    for page_number, section_title, body in merged_segments:
        for chunk_text in chunk_by_sentences(
            body, target_size=target_chunk_size, overlap_sentences=overlap_sentences
        ):
            all_chunks.append(
                Document(
                    page_content=chunk_text,
                    metadata={
                        "book_name": book_name,
                        "page_number": page_number,
                        "chapter": get_chapter(page_number),
                        "section": section_title,
                    },
                )
            )
    return all_chunks


# --- سلول ۹: اجرا ---
pdf_file_path = "/content/drive/MyDrive/ensan-va-iman.pdf"
book_title = "انسان و ایمان"


chunks = process_book(pdf_file_path, book_title, start_page=5, end_page=None)

print(f"\n✅ {len(chunks)} قطعه ساخته شد.")
print("-" * 40)
for c in chunks[:3]:
    print("متادیتا:", c.metadata)
    print(c.page_content[:250])
    print()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
در حال OCR کتاب: انسان و ایمان (67 صفحه)...
  ...10 صفحه پردازش شد
  ...20 صفحه پردازش شد
  ...30 صفحه پردازش شد
  ...40 صفحه پردازش شد
  ...50 صفحه پردازش شد
  ...60 صفحه پردازش شد

✅ 165 قطعه ساخته شد.
----------------------------------------
متادیتا: {'book_name': 'انسان و ایمان', 'page_number': 7, 'chapter': 'انسان و حیوان', 'section': 'انسان و حیوان'}
انسان. خود نوعی حیوان است از این رو با دیگر جانداران مشترکات بسیار دارد اما یک سلسله تفاوتها با هم جنسان خود دارد که او را از جانداران دیگر متمایز ساخته و به او مزیت و تعالی بخشیده و او را بی رقیب ساخته است. جانداران عموما از این مزیت بهره‌مندند که خ

متادیتا: {'book_name': 'انسان و ایمان', 'page_number': 7, 'chapter': 'انسان و حیوان', 'section': 'انسان و حیوان'}
انسان نیز مانند جانداران دیگر یک سلسله خواسته‌ها و مطلوبها دارد و در پرتو آگاهیها و شناختهای خویش برای رسیدن به آن خواسته‌ها و مطلوبها در تلاش است

In [ ]:
!pip install -q langchain-huggingface langchain-chroma sentence-transformers chromadb

In [ ]:

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma


embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [ ]:

import os
os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
)


import shutil, os

PERSIST_DIR = "/content/drive/MyDrive/motahari_chroma_db"
COLLECTION_NAME = "ensan_va_iman_test"


if os.path.exists(PERSIST_DIR):
    shutil.rmtree(PERSIST_DIR)
    print(f"پوشه‌ی قدیمی پاک شد: {PERSIST_DIR}")

vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=PERSIST_DIR,
    collection_name=COLLECTION_NAME,
)

print(f"✅ {len(chunks)} چانک با موفقیت در Chroma ذخیره شد.")

# adaptive retrieval
def cosine_similarity_from_distance(distance):
    """
    چون embedding ها normalize شده‌اند (طول=۱) و Chroma به‌صورت پیش‌فرض
    از فاصله اقلیدسی مربعی (L2²) استفاده می‌کند:
        distance² = 2 − 2×cos_similarity  →  cos_similarity = 1 − (distance/2)
    """
    return 1 - (distance / 2)


def retrieve_context(query, k=5, gap_threshold=0.06, max_results=3, min_similarity=0.35):
    """
    علاوه بر منطق قبلی (فاصله‌ی نسبی/gap)، یک آستانه‌ی مطلق شباهت هم
    اضافه شده: اگر حتی بهترین نتیجه هم به‌اندازه‌ی کافی به سوال شبیه
    نباشد (شباهت کمتر از min_similarity)، یعنی موضوع احتمالاً کاملاً
    خارج از دیتاست فعلی است -> لیست خالی برمی‌گردانیم.

    نکته مهم درباره مقدار ۰.۳۵: این عمداً پایین گذاشته شده. تست عملی
    نشان داد آستانه‌ی بالاتر (مثلاً ۰.۵۵) سوالاتی را هم فیلتر می‌کند
    که از نظر مفهومی نزدیک‌اند ولی مستقیم عین سوال در متن نیامده‌اند
    (مثلاً «آیا هوش مصنوعی می‌تواند ایمان داشته باشد؟») — این دقیقاً
    همان نوع استدلالِ «تعمیم چارچوب فکری به پدیده‌های جدید» است که
    هدف اصلی این پروژه است، نه یک خطا. تشخیص ظریف‌تر «موضوع نزدیک
    است ولی مستقیم پاسخ نمی‌دهد» را به پرامپت (که برایش تقویت شده)
    واگذار می‌کنیم؛ این فیلتر مطلق فقط برای رد سوالاتِ کاملاً
    بی‌ربط (خارج از هر گونه ارتباط موضوعی) نگه داشته شده است.
    """
    results = vectordb.similarity_search_with_score(query, k=k)
    if not results:
        return []

    top_score = results[0][1]
    top_similarity = cosine_similarity_from_distance(top_score)
    if top_similarity < min_similarity:
        return []

    selected = [results[0]]
    for doc, score in results[1:]:
        if len(selected) >= max_results:
            break
        if (score - top_score) <= gap_threshold:
            selected.append((doc, score))
        else:
            break
    return selected


def search(query, k=5, gap_threshold=0.06, max_results=3):
    """نسخه نمایشی retrieve_context برای تست دستی در کولب."""
    results = retrieve_context(query, k=k, gap_threshold=gap_threshold, max_results=max_results)
    print(f"🔎 «{query}» → {len(results)} نتیجه انتخاب شد (از بین {k} کاندید)")
    for i, (doc, score) in enumerate(results, 1):
        sim = cosine_similarity_from_distance(score)
        print(f"--- نتیجه {i} (فاصله خام: {score:.4f} | شباهت: {sim:.1%}) ---")
        print(f"کتاب: {doc.metadata.get('book_name')} | صفحه: {doc.metadata.get('page_number')} | "
              f"فصل: {doc.metadata.get('chapter')} | بخش: {doc.metadata.get('section')}")
        print(doc.page_content)
        print()
    return results


search("تفاوت انسان و حیوان در چیست؟")
print("=" * 60)
search("آگاهی حیوان چه محدودیت‌هایی دارد؟")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

پوشه‌ی قدیمی پاک شد: /content/drive/MyDrive/motahari_chroma_db
✅ 165 چانک با موفقیت در Chroma ذخیره شد.
🔎 «تفاوت انسان و حیوان در چیست؟» → 3 نتیجه انتخاب شد (از بین 5 کاندید)
--- نتیجه 1 (فاصله خام: 0.6164 | شباهت: 69.2%) ---
کتاب: انسان و ایمان | صفحه: 10 | فصل: انسان و حیوان | بخش: ملاك امتیاژ انسان
پس نتیجه می‌گيريم که تفاوت عمده و اساسی انسان با جانداران دیگر که ملاک((انسانیت)) او است و انسانیت وابسته به آن است علم و ایمان است. درباره امتیاز انسان از جانداران دیگر سخن فراوان گفته شده است برخی منکر امتیاز اساسی میان این نوع و سایر انواع هستند» تفاوت آگاهی و شناخت انسان با حیوان را از قبیل تفاوت کمی و حداکثر تفاوت کیفی می‌دانند. نه تفاوت ماهوی همه آن شگفتیها و اهمیتها و عظمتها که نظر فلاسفه بزرگ شرق و غرب را سخت درباره مساله شناخت در انسان جلب کرده است. چندان مورد توجه این گروه واقع نشده است. این گروه انسان را از نظر خواسته‌ها و مطلوبها نیز یک حیوان تمام عیار می‌دانند بدون کوچکترین تفاوتی از این نظر برخی دیگر تفاوت او را در جان داشتن میل و نه درد و نه لذت.

--- نتیجه 2 (فاصله خام: 0.

[(Document(id='0ca8d4ed-4ce6-4a26-b189-f536ca269f57', metadata={'page_number': 8, 'book_name': 'انسان و ایمان', 'chapter': 'انسان و حیوان', 'section': 'شعا عآگاهي و سطح خواسته حیوان'}, page_content='آگاهی حیوان از جهان تنها به وسیله حواس ظاهره است از این رو سطحی و ظاهری است. به درون و روابط درونی اشیاء نفوذ نمی\u200cکند انیا فردی و جزئی است. از کلیت و عمومیت برخوردار نیست ثالثا منطقه\u200cای است. محدود به محیط زیست حیوان است و به خارج محیط زیست او راه پیدا نمی\u200cکند رابعا حالی است. یعنی بسته به زمان حال است. از گذشته و آینده بریده است حیوان نه از تاریخ خود یا جهان آگاه است و نه درباره آینده می\u200cاندیشد و نه تلاشش به آینده تعلق دارد. حبوان از نظر آگاهی. هرگز از چهارچوب ظواهر, فردیت و جزئیت. محیط زیست. زمان حال خارج نمی\u200cگردد.'),
  0.6142832636833191)]

In [ ]:
!pip install -q google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 84.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.13.4 which is incompatible.
google-adk 2.4.0 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.4.0 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.
google-adk 2.4.0 requires starlette<2,>=1.0.1, but you have starlette 0.52.1 which is incompatible.


In [ ]:
!pip install -q ddgs

In [ ]:

from google import genai
from google.genai import types
from google.colab import userdata
from ddgs import DDGS

api_key = userdata.get("Gemeni_API_Key")
client = genai.Client(api_key=api_key)

MODEL_NAME = "gemini-3.1-flash-lite"


#   ابزار جستجوی وب
def web_search(query: str) -> str:
    """
    در وب جستجو می‌کند و چند نتیجه‌ی مرتبط را برمی‌گرداند. این تابع را
    فقط زمانی صدا بزن که سوال درباره‌ی پدیده، رویداد یا مفهومی است که
    در زمان حیات استاد مطهری (درگذشته به سال ۱۳۵۸ شمسی / ۱۹۷۹ میلادی)
    وجود نداشته یا به وضعیت و مسائل روز جامعه مربوط می‌شود (مثل هوش
    مصنوعی، فناوری‌های جدید، رویدادهای اخیر). برای سوالاتی که مستقیماً
    درباره‌ی محتوای کتاب‌های مطهری هستند، از این ابزار استفاده نکن.

    Args:
        query: عبارت جستجو (ترجیحاً کوتاه و مشخص، می‌تواند به فارسی یا
            انگلیسی باشد؛ برای مفاهیم فنی/علمی جدید، انگلیسی نتیجه‌ی
            بهتری می‌دهد)

    Returns:
        متنی شامل چند نتیجه‌ی جستجو (عنوان + خلاصه + لینک) که می‌توان
        از آن برای پاسخ به سوال استفاده کرد.
    """
    print(f"🌐 [ابزار وب‌سرچ فراخوانی شد] پرسمان: «{query}»")

    try:
        results = DDGS().text(query, max_results=4)
    except Exception as e:
        return f"جستجو با خطا مواجه شد: {e}"

    if not results:
        return "نتیجه‌ای برای این جستجو پیدا نشد."

    blocks = []
    for r in results:
        blocks.append(f"عنوان: {r.get('title', '')}\nخلاصه: {r.get('body', '')}\nمنبع: {r.get('href', '')}")
    return "\n\n".join(blocks)


#  ساخت پرامپت پرسونا
PERSONA_SYSTEM_PROMPT = """
تو یک دستیار پژوهشی هستی که فقط بر اساس آثار استاد شهید مرتضی مطهری پاسخ می‌دهی.

⚠️ مهم‌ترین قانون (قبل از هر چیز دیگر رعایتش کن):
قبل از نوشتن پاسخ، از خودت بپرس: «آیا متن‌های زمینه‌ای زیر واقعاً و
مستقیماً به همین سوالِ مشخص پاسخ می‌دهند، یا فقط موضوعی نزدیک/مرتبط
را مطرح می‌کنند؟»
تفاوت این دو حالت را با یک مثال روشن می‌کنم:
  - اگر سوال این باشد «آیا علم و ایمان متضادند یا مکمل؟» و متن فقط
    بگوید «علم و ایمان دو رکن انسانیت‌اند»، این متن TOPIC نزدیکی
    دارد ولی به رابطه‌ی میان آن دو (تضاد/تکامل) پاسخ نمی‌دهد.
    در این حالت حق نداری از پیش خودت استنتاج کنی که پس «مکمل‌اند»؛
    باید صادقانه بگویی که متن‌های موجود این سوال خاص را بررسی
    نکرده‌اند.
  - فقط وقتی می‌توانی نتیجه‌گیری کنی که مقدمات منطقی‌اش واقعاً و به
    وضوح در متن زمینه‌ای آمده باشد، نه اینکه صرفاً موضوعی مشابه را
    لمس کرده باشد.
اگر مطمئن نیستی متن‌ها واقعاً کافی‌اند، ترجیح با «اعتراف به کمبود
منبع» است، نه با ساختن یک پاسخ قانع‌کننده‌ی به‌ظاهر منطقی.

سایر قوانین:
۱. فقط از «متن‌های زمینه‌ای» (context) که در ادامه داده می‌شود استفاده کن.
   هیچ ادعایی نکن که مستقیماً در این متن‌ها نیامده است.
۲. اگر متن‌های زمینه‌ای برای پاسخ به سوال کافی نبودند، صادقانه بگو که
   در منابع موجود پاسخ روشنی برای این سوال پیدا نکردی؛ چیزی از خودت
   نساز. این مهم‌تر از این است که همیشه یک پاسخ کامل و قانع‌کننده
   بدهی.
۳. اگر در میان چند متن زمینه‌ای، بعضی واقعاً به سوال ربط نداشتند،
   نادیده‌شان بگیر و فقط از بخش‌های مرتبط استفاده کن.
۴. لحن پاسخ باید استدلالی، منطقی و آرام باشد؛ شبیه سبک مطهری: ابتدا
   مساله را باز کن، دیدگاه‌های مختلف را در نظر بگیر، بعد با استدلال
   به جمع‌بندی برس. از زبان محاوره‌ای امروزی یا شعارگونه پرهیز کن.
۵. تفکیک صریح داشته باش بین:
   - آنچه مستقیماً نقل یا برداشت از متن مطهری است
   - آنچه تحلیل یا نتیجه‌گیری خودِ تو بر پایه‌ی آن متن است (این را
     با عبارتی مثل «بر اساس این نگاه می‌توان نتیجه گرفت که...» مشخص کن)
     -- ولی این عبارت فقط برای نتیجه‌گیری‌های واقعاً مبتنی بر متن
     است، نه برای پوشاندن حدس‌های بی‌پایه با ظاهر علمی.
۶. در پایان پاسخ، حتماً ارجاع دقیق بده: نام کتاب، شماره صفحه، و اگر
   موجود بود عنوان بخش (هر ارجاع را فقط یک‌بار بیاور، تکراری ننویس).
   مثال: «(انسان و ایمان، ص ۸، بخش: شعاع آگاهی و سطح خواسته حیوان)»
   اگر از ابزار web_search هم استفاده کردی، لینک منابع اینترنتی
   استفاده‌شده را هم در انتهای پاسخ، جدا از ارجاعات کتاب، فهرست کن.
۷. اگر سوال درباره‌ی پدیده، فناوری یا مساله‌ای است که در زمان حیات
   استاد مطهری وجود نداشته یا به وضعیت روز جامعه مربوط می‌شود (مثل
   هوش مصنوعی، رویدادهای اخیر، فناوری‌های جدید)، از ابزار web_search
   برای گرفتن اطلاعات واقعی و به‌روز درباره‌ی خودِ آن پدیده استفاده
   کن. سپس با تکیه بر جهان‌بینی و چارچوب فکری مطهری (از متن‌های
   زمینه‌ای کتاب) درباره‌اش استدلال کن.
   مهم: در پاسخ نهایی صریحاً مشخص کن کدام بخش از اطلاعات جستجوی وب
   آمده (با عبارتی مثل «طبق منابع اینترنتی فعلی...») و کدام بخش
   برداشت/استدلال تو بر پایه‌ی چارچوب مطهری است. هیچ‌وقت این دو منبع
   را با هم قاطی نکن یا وانمود نکن که خودِ مطهری درباره‌ی این پدیده‌ی
   جدید حرف زده است.
"""

def build_context_block(results):
    """نتایج retrieve_context را به یک متن ساختاریافته برای پرامپت تبدیل می‌کند."""
    blocks = []
    for i, (doc, score) in enumerate(results, 1):
        meta = doc.metadata
        ref = f"{meta.get('book_name')}، ص {meta.get('page_number')}"
        if meta.get("section"):
            ref += f"، بخش: {meta.get('section')}"
        blocks.append(f"[متن {i} — منبع: {ref}]\n{doc.page_content}")
    return "\n\n".join(blocks)


#  تابع اصلی پرسش و پاسخ
def ask_motahari(query, k=5, gap_threshold=0.06, max_results=3, temperature=0.4):
    results = retrieve_context(query, k=k, gap_threshold=gap_threshold, max_results=max_results)

    if not results:
        context_block = "(هیچ متن مرتبطی در پایگاه داده‌ی کتاب پیدا نشد.)"
        print("⚠️ توجه: context کتاب خالی است؛ اگر لازم بود مدل باید از web_search استفاده کند.")
    else:
        context_block = build_context_block(results)

    user_prompt = f"""سوال کاربر: {query}

متن‌های زمینه‌ای از آثار استاد مطهری:

{context_block}

با توجه به قوانین بالا، پاسخ را بنویس. اگر لازم بود از ابزار web_search
برای گرفتن اطلاعات به‌روز درباره‌ی پدیده‌ی مطرح‌شده در سوال استفاده کن."""

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=user_prompt,
        config=types.GenerateContentConfig(
            system_instruction=PERSONA_SYSTEM_PROMPT,
            temperature=temperature,
            tools=[web_search],  # automatic function calling: SDK
        ),
    )

    print("💬 پاسخ:\n")
    print(response.text)
    print("\n" + "-" * 50)
    print(f"(بر اساس {len(results)} قطعه‌ی بازیابی‌شده از کتاب)")
    return response.text


ask_motahari("تفاوت انسان و حیوان در چیست؟")

💬 پاسخ:

برای پاسخ به پرسش درباره تفاوت انسان و حیوان، باید ابتدا به این نکته توجه داشت که انسان از نظر زیستی، خود نوعی حیوان محسوب می‌شود و در بسیاری از ویژگی‌های حیات با سایر جانداران مشترک است. با این حال، بر اساس دیدگاه استاد شهید مرتضی مطهری، تفاوت‌هایی وجود دارد که انسان را از سایر جانداران متمایز کرده و به او تعالی بخشیده است.

در ادامه، این تفاوت‌ها را بر اساس متون ارائه شده بررسی می‌کنیم:

### ۱. اشتراکات و تفاوت در شعاع آگاهی
همه جانداران از این ویژگی برخوردارند که خود و جهان خارج را درک می‌کنند و بر اساس این شناخت، برای رسیدن به خواسته‌ها و مطلوب‌های خود تلاش می‌کنند. انسان نیز در این زمینه با سایر جانداران مشترک است. تفاوت در این سطح، صرفاً در «شعاع، وسعت و گستردگی» آگاهی‌ها و شناخت‌هاست؛ یعنی انسان نسبت به حیوانات، دایره وسیع‌تری از آگاهی را داراست.

### ۲. ملاک اساسی و ماهوی (علم و ایمان)
استاد مطهری فراتر از تفاوت‌های کمی در آگاهی، بر وجود یک تفاوت ماهوی تأکید دارند که ملاک اصلی «انسانیت» است. ایشان در این باره می‌فرمایند:
«تفاوت عمده و اساسی انسان با جانداران دیگر که مل

'برای پاسخ به پرسش درباره تفاوت انسان و حیوان، باید ابتدا به این نکته توجه داشت که انسان از نظر زیستی، خود نوعی حیوان محسوب می\u200cشود و در بسیاری از ویژگی\u200cهای حیات با سایر جانداران مشترک است. با این حال، بر اساس دیدگاه استاد شهید مرتضی مطهری، تفاوت\u200cهایی وجود دارد که انسان را از سایر جانداران متمایز کرده و به او تعالی بخشیده است.\n\nدر ادامه، این تفاوت\u200cها را بر اساس متون ارائه شده بررسی می\u200cکنیم:\n\n### ۱. اشتراکات و تفاوت در شعاع آگاهی\nهمه جانداران از این ویژگی برخوردارند که خود و جهان خارج را درک می\u200cکنند و بر اساس این شناخت، برای رسیدن به خواسته\u200cها و مطلوب\u200cهای خود تلاش می\u200cکنند. انسان نیز در این زمینه با سایر جانداران مشترک است. تفاوت در این سطح، صرفاً در «شعاع، وسعت و گستردگی» آگاهی\u200cها و شناخت\u200cهاست؛ یعنی انسان نسبت به حیوانات، دایره وسیع\u200cتری از آگاهی را داراست.\n\n### ۲. ملاک اساسی و ماهوی (علم و ایمان)\nاستاد مطهری فراتر از تفاوت\u200cهای کمی در آگاهی، بر وجود یک تفاوت ماهوی تأکید دارند که ملاک اصلی «انسانیت» است. ایشان در این 

In [ ]:
!pip install -q gradio

In [11]:

import gradio as gr

def gradio_chat(message, history):
    """
    تابع واسط بین Gradio و ask_motahari. چون ask_motahari از قبل کل
    منطق (retrieval + persona + web search) را دارد، اینجا فقط
    پاسخش را برمی‌گردانیم.
    """
    answer = ask_motahari(message)
    if answer is None:
        return "متأسفانه پاسخی تولید نشد. لطفاً دوباره تلاش کنید."
    return answer


demo = gr.ChatInterface(
    fn=gradio_chat,
    title="🕊️ دستیار پژوهشی آثار استاد شهید مطهری",
    description=(
        "یک چالش فکری، اعتقادی یا مساله‌ی روز جامعه مطرح کن. "
        "پاسخ بر اساس آثار استاد شهید مطهری تولید می‌شود و در صورت نیاز "
        "(برای پدیده‌های جدید) از جستجوی وب هم کمک می‌گیرد."
    ),
    examples=[
        "تفاوت انسان و حیوان در چیست؟",
        "آیا علم و ایمان با یکدیگر در تضادند یا مکمل هم هستند؟",
        "طبق این دیدگاه، آیا هوش مصنوعی می‌تونه یک روز «ایمان» داشته باشه؟",
    ],
    theme="soft",
)


demo.launch(debug=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


<IPython.core.display.Javascript object>

Keyboard interruption in main thread... closing server.
